In [1]:
!pip install -q openai datasets scikit-learn tqdm

In [2]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENAI_API_KEY")

    if not API_KEY:
        raise ValueError("Key not found.")

    os.environ["OPENAI_API_KEY"] = API_KEY

except Exception as err:
    print({err})
    raise

try:
    client = OpenAI(api_key=API_KEY)

    test = client.chat.completions.create(
        model="gpt-5",
        messages=[{"role": "user", "content": "What is the climate in Norway?"}],
    )
    print(test.choices[0].message.content)
    print(f"Tokens: {test.usage.total_tokens}")

except Exception as err:
    print({err})
    raise

Norway’s climate is mild for its latitude due to the warm North Atlantic Current, but it varies a lot with coast–inland distance, latitude, and elevation.

- West and southwest coast (e.g., Bergen, Stavanger, Ålesund, Lofoten): Oceanic. Mild, wet, and windy. Winters around 0–6°C, summers 13–17°C. Very high precipitation (often 1500–3000+ mm/year) with frequent storms.
- Eastern and inland valleys (e.g., Oslo, Lillehammer, Innlandet): Continental. Colder, drier winters and warmer summers. Winter commonly −3 to −10°C (can drop below −20°C), summer 20–26°C with occasional heat waves; 500–900 mm/year.
- Northern coastal (e.g., Tromsø, Finnmark coast): Subarctic oceanic. Winters about −3 to +2°C, cool summers 10–15°C, windy with mixed rain/snow; ~700–1200 mm/year.
- Interior north and plateaus (e.g., Finnmarksvidda, Trøndelag interior): Subarctic/continental. Long, cold winters, short cool summers; drier (300–600 mm/year) and large temperature swings.
- Mountains (Scandes, Hardangervidda, J

In [3]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

dataset = load_dataset("ailsntua/QEvasion")
train_df_full = dataset['train'].to_pandas()

validation_size = 750
train_split_df, eval_df = train_test_split(
    train_df_full,
    test_size=validation_size,
    stratify=train_df_full['evasion_label'],
    random_state=42
)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3448 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/308 [00:00<?, ? examples/s]

Train split: 2698 samples
Validation split: 750 samples

Validation label distribution:
evasion_label
Claims ignorance        26
Clarification           20
Declining to answer     31
Deflection              83
Dodging                154
Explicit               229
General                 84
Implicit               106
Partial/half-answer     17
Name: count, dtype: int64


In [4]:
TAXONOMY_TEXT = """
1. Explicit - The information requested is explicitly stated
2. Implicit - The information requested is given, but without being explicitly stated
3. Dodging - Ignoring the question altogether
4. General - The information provided is too general/lacks the requested specificity
5. Deflection - Starts on topic but shifts the focus
6. Partial/half-answer - Offers only a specific component of the requested information
7. Declining to answer - Acknowledge the question but directly or indirectly refusing to answer
8. Claims ignorance - The answerer claims/admits not to know the answer
9. Clarification - Does not provide the requested information and asks for clarification
"""

SYSTEM_MESSAGE = f"""You are an expert in analyzing political discourse and question evasion patterns.
Your task is to classify the type of response provided by an interviewee using the following taxonomy:

{TAXONOMY_TEXT}

In the following examples, you will see 9 demonstration cases (one for each taxonomy class).
Each demonstration consists of 2 messages: one user message with the interview excerpt and question, followed by one assistant message showing the step-by-step reasoning and final classification."""

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": """### Question ###
Do you have your own views about PR at Westminster don't you?

### Answer ###
I do."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Do you have your own views about PR at Westminster?"
2. The answer is: "I do."
3. This directly confirms having views, explicitly answering the yes/no question.
4. The information requested is explicitly stated.

Taxonomy code: Explicit"""
    },

    {
        "role": "user",
        "content": """### Question ###
Are you going to watch television?

### Answer ###
What else is there to do?"""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Are you going to watch television?"
2. The answer is: "What else is there to do?"
3. This rhetorical question suggests they will watch TV (since there's nothing else to do).
4. The information is given indirectly, not explicitly stated.

Taxonomy code: Implicit"""
    },

    {
        "role": "user",
        "content": """### Question ###
Do you like my new dress?

### Answer ###
We are late."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Do you like my new dress?"
2. The answer is: "We are late."
3. The interviewee completely ignores the question about the dress.
4. They change the subject entirely to being late.
5. This is ignoring the question altogether.

Taxonomy code: Dodging"""
    },

    {
        "role": "user",
        "content": """### Question ###
Did you eat the last piece of pie?

### Answer ###
I have to admit that this was a great recipe, I always like it when there are chocolate chips in the dough."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Did you eat the last piece of pie?"
2. The answer acknowledges the pie ("great recipe") but then shifts focus to chocolate chips.
3. The interviewee starts on topic but goes on a tangent without answering whether they ate it.
4. This is deflection - acknowledging the question but shifting focus.

Taxonomy code: Deflection"""
    },

    {
        "role": "user",
        "content": """### Question ###
Did you enjoy the film?

### Answer ###
The directing was great."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Did you enjoy the film?"
2. The answer only addresses the directing, which is one component of a film.
3. The interviewee doesn't address acting, plot, cinematography, or overall enjoyment.
4. This is a partial answer - only addressing one specific component.

Taxonomy code: Partial/half-answer"""
    },

    {
        "role": "user",
        "content": """### Question ###
What's your favorite film?

### Answer ###
Fight Club, Filth, and Hereditary."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks for ONE favorite film (singular).
2. The answer provides THREE films.
3. This makes the desired information unclear - which one is truly the favorite?
4. The reply is too general and lacks the requested specificity.

Taxonomy code: General"""
    },

    {
        "role": "user",
        "content": """### Question ###
The hypothesis I was discussing, wouldn't you regard that as a defeat?

### Answer ###
I am not going to prophesy what will happen."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks for their view on whether something would be a defeat.
2. The answer is: "I am not going to prophesy what will happen."
3. The interviewee directly states they won't answer.
4. They acknowledge the question but refuse to provide the information.

Taxonomy code: Declining to answer"""
    },

    {
        "role": "user",
        "content": """### Question ###
On what precise date did the government order the refit of the HMAS Kanimbla in preparation for its forward deployment to a possible war against Iraq?

### Answer ###
I do not know that date. I will find out and let the House know."""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks for a specific date.
2. The answer is: "I do not know that date."
3. The interviewee admits they don't have the information.
4. This is claiming ignorance - they state they don't know the answer.

Taxonomy code: Claims ignorance"""
    },

    {
        "role": "user",
        "content": """### Question ###
Was it your decision to release the fund?

### Answer ###
You mean the public fund?"""
    },
    {
        "role": "assistant",
        "content": """Let's think step by step:
1. The question asks: "Was it your decision to release the fund?"
2. The answer is: "You mean the public fund?"
3. The interviewee doesn't provide any information about the decision.
4. Instead, they ask for clarification about which fund.
5. This is asking for clarification without giving the requested data.

Taxonomy code: Clarification"""
    }
]

USER_PROMPT_TEMPLATE = """Now, you have to analyze this new case:

### Question ###
{question}

### Answer ###
{interview_answer}

Instructions for your analysis:
1. Think step by step about the relationship between the question and the answer.
2. Provide a brief explanation of your reasoning.
3. You MUST end your response with "Taxonomy code: [Label Name]" exactly as shown in the examples above.

Let's think step by step and provide your classification."""

In [ ]:
from tqdm import tqdm
import time

response_to_label_map = {
    "explicit": "Explicit",
    "implicit": "Implicit",
    "dodging": "Dodging",
    "general": "General",
    "deflection": "Deflection",
    "partial": "Partial/half-answer",
    "partial/half-answer": "Partial/half-answer",
    "partial/half answer": "Partial/half-answer",
    "half-answer": "Partial/half-answer",
    "half answer": "Partial/half-answer",
    "declining": "Declining to answer",
    "declining to answer": "Declining to answer",
    "ignorance": "Claims ignorance",
    "claims ignorance": "Claims ignorance",
    "clarification": "Clarification",
}

predictions = []
api_call_times = []

print(f"Starting GPT-5 prompting on {len(eval_df)} validation examples...\n")

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Processing"):
    question = row['question']
    interview_answer = row['interview_answer']

    user_prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        interview_answer=interview_answer
    )

    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE}
    ] + FEW_SHOT_EXAMPLES + [
        {"role": "user", "content": user_prompt}
    ]

    start_time = time.time()

    try:
        response = client.chat.completions.create(
            model="gpt-5",
            messages=messages,
        )

        api_call_times.append(time.time() - start_time)
        response_text = response.choices[0].message.content.strip()

        predicted_label = None
        if "taxonomy code:" in response_text.lower():
            parts = response_text.lower().split("taxonomy code:")
            if len(parts) > 1:
                candidate = parts[-1].strip().split('\n')[0].strip()
                predicted_label = response_to_label_map.get(candidate)

        if predicted_label is None:
            print(f"\nError at index {idx}: Label not found in response")
            break

        predictions.append(predicted_label)

    except Exception as e:
        print(f"\nError at index {idx}: {e}")
        break

print(f"\n Prompting complete: {len(predictions)} predictions")
print(f"  Total time: {sum(api_call_times)/60:.1f} minutes\n")

results_df = eval_df.copy()
results_df['predicted_label'] = predictions

output_file = "gpt5-fscot-validation-predictions.csv"
results_df.to_csv(output_file, index=False)

In [6]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

y_true = results_df['evasion_label'].tolist()
y_pred = predictions

all_labels = [
    "Clarification",
    "Claims ignorance",
    "Declining to answer",
    "Deflection",
    "Dodging",
    "Explicit",
    "General",
    "Implicit",
    "Partial/half-answer"
]

accuracy = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp) / len(y_true)
macro_f1 = f1_score(y_true, y_pred, average='macro', labels=all_labels, zero_division=0)

print("Subtask 2: Evasion Classification\n")

print(f"Macro F1: {macro_f1:.4f}")

print("\nPer-class metrics:")
print(classification_report(y_true, y_pred, labels=all_labels, zero_division=0))

print("Top 5 confusions:")
cm = confusion_matrix(y_true, y_pred, labels=all_labels)
confusions = []
for i in range(len(all_labels)):
    for j in range(len(all_labels)):
        if i != j and cm[i][j] > 0:
            confusions.append((cm[i][j], all_labels[i], all_labels[j]))

confusions.sort(reverse=True)
for count, true_label, pred_label in confusions[:5]:
    print(f"  {true_label:25s} -> {pred_label:25s}: {count:3d} times")

EVALUATION 1: FINE-GRAINED (9 CLASSES)

Accuracy: 0.4013
Macro F1: 0.3881

Per-class metrics:
                     precision    recall  f1-score   support

      Clarification       0.76      0.65      0.70        20
   Claims ignorance       0.76      0.50      0.60        26
Declining to answer       0.38      0.58      0.46        31
         Deflection       0.19      0.72      0.30        83
            Dodging       0.84      0.20      0.32       154
           Explicit       0.65      0.63      0.64       229
            General       0.32      0.11      0.16        84
           Implicit       0.24      0.08      0.12       106
Partial/half-answer       0.14      0.24      0.18        17

           accuracy                           0.40       750
          macro avg       0.48      0.41      0.39       750
       weighted avg       0.53      0.40      0.39       750

Top 5 confusions:
  Dodging                   -> Deflection               :  89 times
  Implicit              

In [8]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

CLARITY_MAPPING = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "Dodging": "Ambivalent",
    "General": "Ambivalent",
    "Deflection": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply",
}

y_true_clarity = results_df['clarity_label'].tolist()
y_pred_clarity = [CLARITY_MAPPING[pred] for pred in predictions]

accuracy_clarity = sum(1 for yt, yp in zip(y_true_clarity, y_pred_clarity) if yt == yp) / len(y_true_clarity)
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
macro_f1_clarity = f1_score(y_true_clarity, y_pred_clarity, average='macro', labels=clarity_labels, zero_division=0)

print("Subtask 1: Evasion-based Clarity Classification\n")

print(f"Macro F1: {macro_f1_clarity:.4f}")

print("\nPer-class metrics:")
print(classification_report(y_true_clarity, y_pred_clarity, labels=clarity_labels, zero_division=0))


Accuracy: 0.7240
Macro F1: 0.6843

Per-class metrics:
                 precision    recall  f1-score   support

    Clear Reply       0.65      0.63      0.64       229
     Ambivalent       0.78      0.79      0.78       444
Clear Non-Reply       0.61      0.65      0.63        77

       accuracy                           0.72       750
      macro avg       0.68      0.69      0.68       750
   weighted avg       0.72      0.72      0.72       750

Confusion Matrix:

                     |       CR |       AR |      CNR
-------------------------------------------------------
True: CR             |      144 |       79 |        6
True: AR             |       69 |      349 |       26
True: CNR            |        7 |       20 |       50
